In [0]:
#%pip install -r ../requirements.txt

In [0]:
#%pip install --upgrade numpy tensorflow
#%pip install transformers
# %pip install openai datasets torch trl peft bitsandbytes

In [0]:
#%pip install datasets

In [0]:
#%restart_python

In [0]:
from datasets import Dataset
from transformers import AutoTokenizer
import json
from tqdm import tqdm
import os
import mlflow
mlflow.tracing.disable()



In [0]:
# from openai import OpenAI
# import os
# import mlflow
# mlflow.tracing.disable()

# #from transformers import pipeline, AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer


In [0]:

# DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# client = OpenAI(
#   api_key=DATABRICKS_TOKEN,
#   base_url="https://adb-4424763777139025.5.azuredatabricks.net/serving-endpoints"
# )


In [0]:
model_id_databricks = "databricks-meta-llama-3-3-70b-instruct"
model_id = "meta-llama/Llama-3.3-70B-Instruct"
file_name = "cleaned_synthetic_QA_20251017_115344_samples500-1710.json"

# Reformat dataset for model

Why you do need apply_chat_template for fine-tuning

For LoRA fine-tuning, you’re not using the API — you’re loading the model weights locally or in Databricks notebooks (with transformers, trl, etc.) and training a new model version.

That model expects data that looks exactly like this inside each example:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What are common causes of knee pain?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Knee pain is often caused by overuse, injury, or underlying conditions...<|eot_id|><|end_of_text|>


To generate that structure correctly, you use:

tokenizer.apply_chat_template(messages, tokenize=False)

In [0]:
from huggingface_hub import login
hf_token="pippo"
login(token=hf_token)

In [0]:
%env HUGGINGFACE_HUB_TOKEN=hf_token


In [0]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_size="right"



In [0]:
def transform_to_dataset(dataset):
    return Dataset.from_list(dataset) 
    
def load_dataset(json_file_name):
    with open(json_file_name, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    return dataset



In [0]:
test_size = 0.3
dataset_json = load_dataset(file_name)[:100]
dataset = transform_to_dataset(dataset_json)
dataset_splited = dataset.train_test_split(test_size=test_size, seed=42, shuffle=True)
train_dataset = dataset_splited["train"]

test_val_dataset = dataset_splited["test"].train_test_split(test_size=0.5, seed=42, shuffle=True)   
test_dataset = test_val_dataset["train"]
val_dataset = test_val_dataset["test"]

In [0]:

def format_sample(example):
    messages  = [
        {"role": "system", "content": "You are a helpful physiotherapy assistant."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# dataset = load_dataset(file_name)
# formatted_dataset = [format_sample(e) for e in test_dataset]


In [0]:
train_dataset = format_sample(train_dataset)
test_dataset = format_sample(test_dataset)
val_dataset = format_sample(val_dataset)

#Load Based Model

In [0]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [0]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM, PeftModel
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling



In [0]:
#%pip install -U bitsandbytes

In [0]:
#
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM
import torch 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype='float16',
    bnb_4bit_use_double_quant=True
)

In [0]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16", 
    quantization_config=bnb_config,
)

model.config.use_cache=False
model.config.pretraining_tp=1

In [0]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',# the other type is the MaskedLM -> One is masked language modeling in which randomly some tokens are masked 
    #and then model try to predict that token.
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)


In [0]:
from transformers import TrainingArguments

In [0]:
eval_steps=200
training_config = SFTConfig(
    output_dir="./finetuned-llama3.3-70B",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,               # Increased from 1 for 4k samples
    max_length=2048,
    learning_rate=1e-4,
    logging_steps=20,                 # Log training info every 20 steps
    save_steps=eval_steps*3,                   # Save checkpoints every 500 steps
    save_total_limit=2,               # Keep only 2 latest checkpoints
    eval_strategy="steps",            # Evaluate periodically during training
    eval_steps=eval_steps,                   # Evaluate every 200 steps
    load_best_model_at_end=True,      # Load best checkpoint based on metric
    metric_for_best_model="eval_loss",# Use eval_loss to select best model
    greater_is_better=False,          # Lower loss is better
    #max_steps=None,    
    max_steps=2,                # Optional: set max_steps to stop early
    fp16=True,                        # Mixed precision training
    weight_decay=0.01,                # Regularization to prevent overfitting
    gradient_checkpointing=True,      # Save memory for large models
    report_to="none"                  # Disable WandB/MLflow logging if not needed
)

# ----------------------------
# 2️⃣ Trainer Setup
# ----------------------------
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_config,
    train_dataset=train_dataset,           # Your training dataset
    eval_dataset=val_dataset,              # Validation dataset
    dataset_text_field="text",             # Field containing preformatted chat text
    peft_config=peft_config                # LoRA configuration
)



In [0]:
# ----------------------------
# 3️⃣ Start Training
# ----------------------------
trainer.train()

In [0]:
model.save_pretrained(f"{model_id}-qlora")

In [0]:
FBIREVBKENV

In [0]:
training_config = SFTConfig(
    output_dir="./finetuned-llama3.3-70B",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_seq_length=2048,
    learning_rate=1e-4,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_config,
    train_dataset=dataset,
    dataset_text_field="text"  # <---- the field with formatted text
)

trainer.train()
